In [2]:
import numpy as np
import pandas as pd

In [3]:
hydrate=pd.read_csv('input/s2-hydrate.csv',index_col=0)
cages=pd.read_csv('input/s2-cages.csv')
thf=pd.read_csv('input/thf.csv')

In [4]:
hydrate_ox=pd.DataFrame()
hydrate_ox[['X','Y','Z']]=hydrate[['OX','OY','OZ']]
hydrate_ox.insert(0,'ELEMENT','OW')
hydrate_hy=pd.DataFrame()
hydrate_hy[['X','Y','Z']]=hydrate[['H1X','H1Y','H1Z']]
hydrate_hy.insert(0, 'ELEMENT', 'HW1')
hydrate_hz=pd.DataFrame()
hydrate_hz[['X','Y','Z']]=hydrate[['H2X','H2Y','H2Z']]
hydrate_hz.insert(0, 'ELEMENT', 'HW2')

In [5]:
hydrate_total = pd.DataFrame(columns=['ELEMENT', 'X', 'Y', 'Z', 'RESIDUE_NUM'])
residue_num = 1
for row in range(len(hydrate_ox.index)):
    water = pd.concat([hydrate_ox.iloc[row], hydrate_hy.iloc[row], hydrate_hz.iloc[row]], axis=1, ignore_index=True).T
    water['RESIDUE_NUM'] = residue_num
    hydrate_total = pd.concat([hydrate_total, water], axis=0, ignore_index=True)
    residue_num += 1
hydrate_total['MOL'] = 'HOH'
hydrate_total

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,OW,5.32179,5.32179,8.5662,2,HOH
4,HW1,5.53914,5.52632,7.65672,2,HOH
...,...,...,...,...,...,...
403,HW1,4.86407,13.52592,13.58037,135,HOH
404,HW2,3.79358,12.45544,13.57796,135,HOH
405,OW,12.98252,4.31665,12.9933,136,HOH
406,HW1,12.44593,3.72957,13.52588,136,HOH


In [6]:
tip4p_hoh = hydrate_total.loc[hydrate_total['RESIDUE_NUM'] == 1]

In [7]:
def virtual_coord(x1, x2, x3):
    """
    Returns the virtual coordinate (or 4th point) for a TIP4P water using the following alogrithm:
    x4 = x1 + a*(x2-x1) + b*(x3-x1)
    where a,b = 0.128012065

    Refer to OPLSAA TIP4P itp file in GROMACS for more information

    Author: Gabe Miles
    """
    a = b = 0.128012065
    return x1 + a * (x2 -x1) + b * (x3 - x1)

In [8]:
tip4p_hoh.loc[len(tip4p_hoh.index)] = ['MW', 0, 0, 0, 1, 'HOH']

/tmp/ipykernel_14288/324350337.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hoh.loc[len(tip4p_hoh.index)] = ['MW', 0, 0, 0, 1, 'HOH']


In [9]:
tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'MW'] = virtual_coord(tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'OW'][0], 
                                                                 tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'HW1'][1],
                                                                 tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'HW2'][2])

tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'MW'] = virtual_coord(tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'OW'][0], 
                                                                 tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'HW1'][1],
                                                                 tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'HW2'][2])

tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] == 'MW'] = virtual_coord(tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] == 'OW'][0], 
                                                                 tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] == 'HW1'][1],
                                                                 tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] == 'HW2'][2])
tip4p_hoh

/tmp/ipykernel_14288/462234224.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'MW'] = virtual_coord(tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'OW'][0],
/tmp/ipykernel_14288/462234224.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'MW'] = virtual_coord(tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'OW'][0],
/tmp/ipykernel_14288/462234224.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/i

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH


In [10]:
%matplotlib qt5

In [11]:
import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(xs=tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] != 'MW'].values,
           ys=tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] != 'MW'].values,
           zs=tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] != 'MW'].values,
           c='red')
ax.scatter(xs=tip4p_hoh['X'].loc[tip4p_hoh['ELEMENT'] == 'MW'].values,
           ys=tip4p_hoh['Y'].loc[tip4p_hoh['ELEMENT'] == 'MW'].values,
           zs=tip4p_hoh['Z'].loc[tip4p_hoh['ELEMENT'] == 'MW'].values,
           c='blue')


Figure shows that the 4th point in the model is in the correct location. Ready to move forward and add 4th point to total model.

In [12]:
tip4p_hydrate = pd.DataFrame(columns=['ELEMENT', 'X', 'Y', 'Z', 'RESIDUE_NUM'])
mw_x_coord = virtual_coord(hydrate_ox['X'], hydrate_hy['X'], hydrate_hz['X'])
mw_y_coord = virtual_coord(hydrate_ox['Y'], hydrate_hy['Y'], hydrate_hz['Y'])
mw_z_coord = virtual_coord(hydrate_ox['Z'], hydrate_hy['Z'], hydrate_hz['Z'])

hydrate_mw = pd.DataFrame({'X': mw_x_coord,
                           'Y': mw_y_coord,
                           'Z': mw_z_coord})
hydrate_mw.insert(0,'ELEMENT','MW')
hydrate_mw

,ELEMENT,X,Y,Z
1,MW,6.021258,6.023186,5.865343
2,MW,5.452710,5.300656,8.496106
3,MW,7.681844,4.168606,9.780210
4,MW,9.508757,4.288440,7.644088
5,MW,8.616587,5.338877,5.181544
...,...,...,...,...
132,MW,0.000000,8.804999,8.655000
133,MW,8.655000,0.000000,8.505001
134,MW,13.132513,12.992332,4.317725
135,MW,4.327824,12.992666,13.143288


In [13]:
tip4p_hydrate_total = pd.DataFrame(columns=['ELEMENT', 'X', 'Y', 'Z', 'RESIDUE_NUM'])
residue_num = 1
for row in range(len(hydrate_ox.index)):
    water = pd.concat([hydrate_ox.iloc[row], hydrate_hy.iloc[row], hydrate_hz.iloc[row], hydrate_mw.iloc[row]],
                       axis=1, ignore_index=True).T
    water['RESIDUE_NUM'] = residue_num
    tip4p_hydrate_total = pd.concat([tip4p_hydrate_total, water], axis=0, ignore_index=True)
    residue_num += 1
tip4p_hydrate_total['MOL'] = 'HOH'
tip4p_hydrate_total

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
539,MW,4.327824,12.992666,13.143288,135,HOH
540,OW,12.98252,4.31665,12.9933,136,HOH
541,HW1,12.44593,3.72957,13.52588,136,HOH
542,HW2,13.51643,3.73198,12.4554,136,HOH


In [14]:
# Now that I have a TIP4P hydrate, lets insert my THF into the large cages
large_cages = cages[cages['TYPE'].str.contains('Large')]
thf_center = thf[['X', 'Y', 'Z']].mean()
# Set center to (0,0,0)
thf[['X','Y','Z']] = thf[['X','Y','Z']] - thf_center
# Update to new center
thf['RESIDUE_NUM'] = -1
thf['MOL'] = 'THF'
large_cages

,TYPE,X,Y,Z
16,Large 1,8.651,8.655,8.654
17,Large 2,4.322,4.327,12.984
18,Large 3,12.991,4.327,4.326
19,Large 4,4.319,12.983,4.326
20,Large 5,12.988,12.983,12.984
21,Large 6,0.004,0.000,8.655
22,Large 7,8.655,0.000,0.001
23,Large 8,0.000,8.655,0.000


In [15]:
from scipy.spatial.transform import Rotation as R

# The goal here is to randomize the orientation of THF in the hydrate to prevent
# the creation of a dipole. 
# This stack overflow post was helpful: https://stackoverflow.com/questions/14607640/rotating-a-vector-in-3d-space
thf_rot = thf.copy()
# Get the center of the THF to 0 (origin)
thf_rot[['X','Y','Z']] = thf_rot[['X','Y','Z']] - thf_center

# Now do a simple rotation around the Z axis
r = R.from_euler('zyx', [90,45,30], degrees=True)
# np.vstack(thf_rot[['X','Y','Z']].iloc[0].values)
# np.vstack returns the column vector of 'zyx', and the '@' does matrix multiplication
tmp = r.as_matrix() @ np.vstack(thf_rot[['X','Y','Z']].iloc[0].values)
# tmp[:,[0]].T[0]

In [16]:
from random import randint
from scipy.spatial.transform import Rotation as R

def rotate_thf(thf):
    """
    This function randomizes the orientation of THF in space. It does this with a transform matrix
    provided by the scipy Rotation package. By using this package, THF can be rotated around it's
    center in 3D space, and not just around an axis. This funtion also provides random rotations,
    with a random number of degrees in each direction.

    For more information on the theory, visit:
    https://stackoverflow.com/questions/14607640/rotating-a-vector-in-3d-space

    Input: THF dataframe that includes columns for 'X', 'Y', and 'Z'

    Output: Identical THF dataframe with updated coordinates given a random rotation in 3D space

    Author: Gabe Miles
    """
    x,y,z = randint(0,360), randint(0,360), randint(0,360)
    r = R.from_euler('zyx', [x,y,z], degrees=True)
    tmp = np.empty((0,3), int)
    for i in range(len(thf_rot.index)):
        # np.vstack returns the column vector of 'zyx', and the '@' does matrix multiplication
        new_xyz = r.as_matrix() @ np.vstack(thf[['X','Y','Z']].iloc[i].values)
        tmp = np.append(tmp, new_xyz.T, axis=0)

    rot_thf = pd.DataFrame({'ELEMENT': thf['ELEMENT'],
                        'X': tmp[:,[0]].T[0],
                        'Y': tmp[:,[1]].T[0],
                        'Z': tmp[:,[2]].T[0],
                        'RESIDUE_NUM': thf['RESIDUE_NUM'],
                        'MOL': thf['MOL']})
    
    return rot_thf

In [26]:
# rotate_thf(thf)
thf

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,C00,0.647,-1.164769,0.116231,-1,THF
1,C01,-0.759,-0.711769,-0.208769,-1,THF
2,C02,-0.735,0.730231,0.227231,-1,THF
3,C03,0.674,1.148231,-0.131769,-1,THF
4,O04,1.496,-0.016769,-0.015769,-1,THF
5,H05,0.995,-1.957769,-0.550769,-1,THF
6,H06,0.721,-1.517769,1.150231,-1,THF
7,H07,-1.525,-1.309769,0.292231,-1,THF
8,H08,-0.929,-0.769769,-1.290769,-1,THF
9,H09,-0.877,0.792231,1.313231,-1,THF


In [43]:
f = open("output/thf.pdb", "w")

f.write("CRYST1   1.0000   1.0000   1.0000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(thf.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = thf['ELEMENT'].iloc[i]
    resname = thf['MOL'].iloc[i]
    resnum = 0
    x = thf['X'].iloc[i]
    y = thf['Y'].iloc[i]
    z = thf['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()

In [42]:
import matplotlib.pyplot as plt
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(xs=thf['X'].loc[thf['ELEMENT'] != 'C00'].values,
           ys=thf['Y'].loc[thf['ELEMENT'] != 'C00'].values,
           zs=thf['Z'].loc[thf['ELEMENT'] != 'C00'].values,
           c='red')
ax.scatter(xs=thf['X'].loc[thf['ELEMENT'] == 'C00'].values,
           ys=thf['Y'].loc[thf['ELEMENT'] == 'C00'].values,
           zs=thf['Z'].loc[thf['ELEMENT'] == 'C00'].values,
           c='blue')
ax.scatter(xs=thf['X'].loc[thf['ELEMENT'] == 'H0A'].values,
           ys=thf['Y'].loc[thf['ELEMENT'] == 'H0A'].values,
           zs=thf['Z'].loc[thf['ELEMENT'] == 'H0A'].values,
           c='green')

In [ ]:
# This loop recenters THF and places it in all 8 large cages of the sII hydrate
thf_hydrate_randomized = tip4p_hydrate_total.copy()
for i in range(len(large_cages.index)):
    rot_thf = rotate_thf(thf)
    dxyz = thf_center - large_cages[['X','Y','Z']].iloc[i]
    thf_translated = rot_thf.copy()
    thf_translated[['X','Y','Z']] = thf_translated[['X','Y','Z']] - dxyz
    thf_translated['RESIDUE_NUM'] = residue_num
    thf_hydrate_randomized = pd.concat([thf_hydrate_randomized, thf_translated], ignore_index=True)
    residue_num += 1

thf_hydrate_randomized

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
643,H08,-1.250905,8.270317,-0.62679,144,THF
644,H09,0.608276,8.64652,1.744978,144,THF
645,H0A,-0.818567,9.620381,1.340466,144,THF
646,H0B,1.677118,10.411516,0.554543,144,THF


In [ ]:
# Write unit cell pdb
f = open("output/tip4p_thf_hydrate_randomized.pdb", "w")

f.write("CRYST1   17.310   17.310   17.310  90.00  90.00  90.00 P 1           1\n")
for i in range(len(thf_hydrate_randomized.index)):
    #Collect PDB variables
    id = str(i)
    atom = thf_hydrate_randomized['ELEMENT'].iloc[i]
    resname = thf_hydrate_randomized['MOL'].iloc[i]
    resnum = thf_hydrate_randomized['RESIDUE_NUM'].iloc[i]
    x = thf_hydrate_randomized['X'].iloc[i]
    y = thf_hydrate_randomized['Y'].iloc[i]
    z = thf_hydrate_randomized['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s}  {:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()

So previously I have made my unit cell w/ THF inserted already, and then I have simply stacked the unit cells together to achieve my overall hydrate structure. That worked great when I didn't care about the orientation of THF inside the hydrate. Now I do. So, I will create the the overall hydrate strucure w/o THF inserted, and in the process I will do all the same transformations with the large cage centers. Once I have made my large hydrate, I will then place the THF (randomized orientations) into the large hydrate.

In [ ]:
#Make the box 3 x 3
cell_size = 17.30
num_cells = 3
coord_offset = [[-cell_size, 0, cell_size],[0,0,cell_size],[cell_size, 0, cell_size],
                [-cell_size, 0, 0],[cell_size, 0, 0],
                [-cell_size, 0, -cell_size],[0, 0, -cell_size],[cell_size, 0, -cell_size]]

In [ ]:
# Create 3x3 plane crystal
tip4p_hydrate_3_plane = tip4p_hydrate_total.copy()
for offset in coord_offset:
    # For hydrate
    residue_num = tip4p_hydrate_3_plane['RESIDUE_NUM'].iloc[len(tip4p_hydrate_3_plane.index) - 1]
    tip4p_hydrate_unit = tip4p_hydrate_total.copy()
    tip4p_hydrate_unit[['X','Y','Z']] = tip4p_hydrate_unit[['X','Y','Z']] + offset
    tip4p_hydrate_unit['RESIDUE_NUM'] = tip4p_hydrate_unit['RESIDUE_NUM'] + residue_num
    tip4p_hydrate_3_plane = pd.concat([tip4p_hydrate_3_plane, tip4p_hydrate_unit], axis=0, ignore_index=True)


In [ ]:
cages_3_plane = large_cages.copy()
for offset in coord_offset:
    # For hydrate
    large_cages_unit = large_cages.copy()
    large_cages_unit[['X','Y','Z']] = large_cages_unit[['X','Y','Z']] + offset
    cages_3_plane = pd.concat([cages_3_plane, large_cages_unit], axis=0, ignore_index=True)

In [ ]:
# Stack 3x3x3 crystal
tip4p_hydrate_3_3 = tip4p_hydrate_3_plane.copy()
for i in range(1, num_cells):
    residue_num = tip4p_hydrate_3_3['RESIDUE_NUM'].iloc[len(tip4p_hydrate_3_3.index) - 1]
    tip4p_hydrate_unit = tip4p_hydrate_3_plane.copy()
    tip4p_hydrate_unit['Y'] = tip4p_hydrate_unit['Y'] + (cell_size * i)
    tip4p_hydrate_unit['RESIDUE_NUM'] = tip4p_hydrate_unit['RESIDUE_NUM'] + residue_num
    tip4p_hydrate_3_3 = pd.concat([tip4p_hydrate_3_3, tip4p_hydrate_unit], axis=0, ignore_index=True)
tip4p_hydrate_3_3

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
14683,MW,21.627824,47.592666,-4.156712,3671,HOH
14684,OW,30.28252,38.91665,-4.3067,3672,HOH
14685,HW1,29.74593,38.32957,-3.77412,3672,HOH
14686,HW2,30.81643,38.33198,-4.8446,3672,HOH


In [ ]:
# Stack 3x3x3 crystal
cages_3_3 = cages_3_plane.copy()
for i in range(1, num_cells):
    cages_unit = cages_3_plane.copy()
    cages_unit['Y'] = cages_unit['Y'] + (cell_size * i)
    cages_3_3 = pd.concat([cages_3_3, cages_unit], axis=0, ignore_index=True)
cages_3_3

,TYPE,X,Y,Z
0,Large 1,8.651,8.655,8.654
1,Large 2,4.322,4.327,12.984
2,Large 3,12.991,4.327,4.326
3,Large 4,4.319,12.983,4.326
4,Large 5,12.988,12.983,12.984
...,...,...,...,...
211,Large 4,21.619,47.583,-12.974
212,Large 5,30.288,47.583,-4.316
213,Large 6,17.304,34.600,-8.645
214,Large 7,25.955,34.600,-17.299


In [ ]:
# This loop recenters THF and places it in all 8 large cages of the sII hydrate
thf_hydrate_randomized_3_3 = tip4p_hydrate_3_3.copy()
for i in range(len(cages_3_3.index)):
    rot_thf = rotate_thf(thf)
    dxyz = thf_center - cages_3_3[['X','Y','Z']].iloc[i]
    thf_translated = rot_thf.copy()
    thf_translated[['X','Y','Z']] = thf_translated[['X','Y','Z']] - dxyz
    thf_translated['RESIDUE_NUM'] = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[len(thf_hydrate_randomized_3_3.index) - 1] + 1
    thf_hydrate_randomized_3_3 = pd.concat([thf_hydrate_randomized_3_3, thf_translated], ignore_index=True)
    residue_num += 1

thf_hydrate_randomized_3_3

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
17491,H08,16.429465,43.442814,-18.560607,3888,THF
17492,H09,17.456326,44.143122,-15.789598,3888,THF
17493,H0A,17.267856,45.242906,-17.169043,3888,THF
17494,H0B,19.675219,44.055792,-16.651957,3888,THF


In [ ]:
f = open("output/tip4p_thf_hydrate_randomized_3_3.pdb", "w")

f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(thf_hydrate_randomized_3_3.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = thf_hydrate_randomized_3_3['ELEMENT'].iloc[i]
    resname = thf_hydrate_randomized_3_3['MOL'].iloc[i]
    resnum = thf_hydrate_randomized_3_3['RESIDUE_NUM'].iloc[i]
    x = thf_hydrate_randomized_3_3['X'].iloc[i]
    y = thf_hydrate_randomized_3_3['Y'].iloc[i]
    z = thf_hydrate_randomized_3_3['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()

In [ ]:
f = open("output/tip4p_hydrate_3_3.pdb", "w")

f.write("CRYST1   52.000   52.000   52.000  90.00  90.00  90.00 P 1           1\n")
for i in range(len(tip4p_hydrate_3_3.index)):
    #Collect PDB variables
    id = str(i)
    chainid = 'A'
    atom = tip4p_hydrate_3_3['ELEMENT'].iloc[i]
    resname = tip4p_hydrate_3_3['MOL'].iloc[i]
    resnum = tip4p_hydrate_3_3['RESIDUE_NUM'].iloc[i]
    x = tip4p_hydrate_3_3['X'].iloc[i]
    y = tip4p_hydrate_3_3['Y'].iloc[i]
    z = tip4p_hydrate_3_3['Z'].iloc[i]

    # Create PDB entry
    line = "{:6s}{:5d} {:^4s} {:3s} {:1s}{:4d}    {:8.3f}{:8.3f}{:8.3f}{:6.2f}{:6.2f}\n".format(
           "ATOM", i+1, atom, resname, chainid, resnum, x, y, z, 0., 0.)
    f.write(line)

f.close()

In [ ]:
# Compute the dipole moment of the water
tip4p_hydrate_3_3

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL
0,OW,5.92487,5.92487,5.92487,1,HOH
1,HW1,5.75998,6.83792,5.68955,1,HOH
2,HW2,6.84272,5.77984,5.69518,1,HOH
3,MW,6.021258,6.023186,5.865343,1,HOH
4,OW,5.32179,5.32179,8.5662,2,HOH
...,...,...,...,...,...,...
14683,MW,21.627824,47.592666,-4.156712,3671,HOH
14684,OW,30.28252,38.91665,-4.3067,3672,HOH
14685,HW1,29.74593,38.32957,-3.77412,3672,HOH
14686,HW2,30.81643,38.33198,-4.8446,3672,HOH


In [ ]:
tip4p_hydrate_3_3['CHARGE'] = 0.
tip4p_hydrate_3_3['CHARGE'].loc[tip4p_hydrate_3_3['ELEMENT'] == 'HW1'] = 0.52 
tip4p_hydrate_3_3['CHARGE'].loc[tip4p_hydrate_3_3['ELEMENT'] == 'HW2'] = 0.52  
tip4p_hydrate_3_3['CHARGE'].loc[tip4p_hydrate_3_3['ELEMENT'] == 'MW'] = -1.04 
tip4p_hydrate_3_3

/tmp/ipykernel_34316/3565793444.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hydrate_3_3['CHARGE'].loc[tip4p_hydrate_3_3['ELEMENT'] == 'HW1'] = 0.52
/tmp/ipykernel_34316/3565793444.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hydrate_3_3['CHARGE'].loc[tip4p_hydrate_3_3['ELEMENT'] == 'HW2'] = 0.52
/tmp/ipykernel_34316/3565793444.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tip4p_hydrate_3_3['CHARGE'].loc

,ELEMENT,X,Y,Z,RESIDUE_NUM,MOL,CHARGE
0,OW,5.92487,5.92487,5.92487,1,HOH,0.00
1,HW1,5.75998,6.83792,5.68955,1,HOH,0.52
2,HW2,6.84272,5.77984,5.69518,1,HOH,0.52
3,MW,6.021258,6.023186,5.865343,1,HOH,-1.04
4,OW,5.32179,5.32179,8.5662,2,HOH,0.00
...,...,...,...,...,...,...,...
14683,MW,21.627824,47.592666,-4.156712,3671,HOH,-1.04
14684,OW,30.28252,38.91665,-4.3067,3672,HOH,0.00
14685,HW1,29.74593,38.32957,-3.77412,3672,HOH,0.52
14686,HW2,30.81643,38.33198,-4.8446,3672,HOH,0.52


In [ ]:
x_dipole = (tip4p_hydrate_3_3['X'] * tip4p_hydrate_3_3['CHARGE']).sum()
y_dipole = (tip4p_hydrate_3_3['X'] * tip4p_hydrate_3_3['CHARGE']).sum()
z_dipole = (tip4p_hydrate_3_3['X'] * tip4p_hydrate_3_3['CHARGE']).sum()

print("X dipole: {:.5f}".format(x_dipole))
print("Y dipole: {:.5f}".format(y_dipole))
print("Z dipole: {:.5f}".format(z_dipole))

X dipole: 0.00240
Y dipole: 0.00240
Z dipole: 0.00240


So the dipole moment is not _exactly_ 0, but it is pretty close. I can try annealing to see if I can get it closer to zero. I ran an energy minimization step, and the hydrate itself is already at a minimum. So now I'll try to do that annealing process Oliviero talked about.